# Python Commands Reference

Quick reference for Python, NumPy, and SymPy commands used in this project.  
One entry per command — what it does and a minimal example. No theory.

---

## Contents

1. [Converting a SymPy Matrix to a NumPy Array](#sympy-to-numpy)
2. [Flattening a Matrix to a 1D Vector](#flatten)
3. [Hashing a File with hashlib](#hashlib)
4. [Saving and Loading Python Objects with pickle](#pickle)
5. [Working with File Paths using pathlib](#pathlib)

---

<a id='sympy-to-numpy'></a>
## 1. Converting a SymPy Matrix to a NumPy Array

**Goal:** evaluate all symbolic expressions to floats so NumPy tools (like `eigvalsh`) can work on the result.

```python
import numpy as np

# M is a SymPy Matrix with symbolic entries
M_num = np.array(M.evalf(), dtype=float)
```

- `.evalf()` — evaluates every element to a floating-point number
- `np.array(..., dtype=float)` — converts the result into a NumPy array

Use this any time you need to go from symbolic → numerical for testing or simulation.

---

<a id='flatten'></a>
## 2. Flattening a Matrix to a 1D Vector

**Goal:** convert a 2D array into a single 1D array so it can be concatenated with other vectors or passed to functions like `np.linalg.norm`.

```python
import numpy as np

R = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]])

R.flatten()  # returns array([1, 2, 3, 4, 5, 6, 7, 8, 9])
```

Reads left-to-right, row by row. Common use in IK — combining a rotation error matrix with a position error vector into one flat error signal:

```python
error = np.concatenate([pos_error, rot_error.flatten()])
```

---

<a id='hashlib'></a>
## 3. Hashing a File with hashlib

**Goal:** produce a short fixed-length string that uniquely identifies the contents of a file. If the file changes, the hash changes.

```python
import hashlib

with open("configs/robots/example_6dof.yaml", "rb") as f:
    file_hash = hashlib.md5(f.read()).hexdigest()

print(file_hash)  # e.g. "a3f1c9d82b..."
```

- `"rb"` — open in **binary** read mode (required for hashing — text mode can alter line endings)
- `hashlib.md5(...)` — creates an MD5 hash object from the bytes
- `.hexdigest()` — returns the hash as a readable hex string

Use this to build a cache key: if the YAML changes, the hash changes and the old cache file is ignored.

**Reference:** https://docs.python.org/3/library/hashlib.html

---

<a id='pickle'></a>
## 4. Saving and Loading Python Objects with pickle

**Goal:** serialise a Python object (like a tuple of SymPy expressions) to a file on disk, then load it back in a later session.

```python
import pickle

# Saving — open in binary write mode
with open("cache/my_robot.pkl", "wb") as f:
    pickle.dump((M, C, g, theta_syms, theta_dot_syms), f)

# Loading — open in binary read mode
with open("cache/my_robot.pkl", "rb") as f:
    M, C, g, theta_syms, theta_dot_syms = pickle.load(f)
```

- `"wb"` / `"rb"` — binary write / binary read (pickle files are binary, not text)
- `pickle.dump(obj, f)` — writes the object into the open file
- `pickle.load(f)` — reads and reconstructs the object from the file
- The tuple is unpacked directly on load — same as any Python tuple assignment

SymPy expressions are picklable, so `M`, `C`, `g`, and symbol lists can all be stored this way.

**Reference:** https://docs.python.org/3/library/pickle.html

---

<a id='pathlib'></a>
## 5. Working with File Paths using pathlib

**Goal:** build file paths, check if files exist, and create directories — without manually joining strings.

```python
from pathlib import Path

# Build a path
cache_dir = Path("cache")
cache_file = cache_dir / "example_6dof_a3f1c9d82b.pkl"

# Check if a file exists
if cache_file.exists():
    print("cache hit")

# Create a directory (and any missing parents) — does nothing if it already exists
cache_dir.mkdir(parents=True, exist_ok=True)
```

- `Path("cache") / "filename.pkl"` — the `/` operator joins path segments (works on all operating systems)
- `.exists()` — returns `True` if the file or directory is present on disk
- `.mkdir(parents=True, exist_ok=True)` — creates the directory; `exist_ok=True` prevents an error if it already exists

**Reference:** https://docs.python.org/3/library/pathlib.html

---